# Chatbot using Hugging Face Transformers

**Objective:** Build a conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

**Model Used:** DialoGPT (Microsoft's dialogue generation model)

## 1. Install and Import Required Libraries

In [1]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random

ModuleNotFoundError: No module named 'torch'

## 2. Load Pre-trained Transformer Model

In [ ]:
# Load pre-trained DialoGPT model and tokenizer
# DialoGPT is a transformer model trained on conversational data
model_name = "microsoft/DialoGPT-medium"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)

## 3. Define Chatbot Class

In [ ]:
class Chatbot:
    """
    A conversational chatbot class using DialoGPT transformer model.
    Maintains conversation history for context-aware responses.
    """
    
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.chat_history_ids = None  # Stores conversation history
        
    def generate_response(self, user_input):
        """
        Generate a response from the chatbot based on user input.
        
        Args:
            user_input (str): The user's input text
            
        Returns:
            str: The chatbot's generated response
        """
        # Encode user input with special tokens
        # return_lang_style adds bos (beginning of sentence) and eos (end of sentence) tokens
        new_input_ids = self.tokenizer.encode(
            user_input + self.tokenizer.eos_token, 
            return_tensors="pt"
        ).to(self.device)
        
        # Append user input to chat history
        if self.chat_history_ids is not None:
            bot_input_ids = torch.cat([self.chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids
        
        # Limit conversation history to prevent memory overflow
        # Keep last 500 tokens for context
        if bot_input_ids.shape[1] > 500:
            bot_input_ids = bot_input_ids[:, -500:]
        
        # Generate response using the model
        self.chat_history_ids = self.model.generate(
            bot_input_ids,
            max_new_tokens=100,          # Maximum tokens to generate
            pad_token_id=self.tokenizer.eos_token_id,
            do_sample=True,              # Enable sampling for diverse responses
            top_k=50,                    # Limit to top 50 tokens
            top_p=0.95,                  # Nucleus sampling
            temperature=0.8,             # Control randomness (lower = more focused)
            num_return_sequences=1
        )
        
        # Decode the generated response
        response = self.tokenizer.decode(
            self.chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )
        
        return response
    
    def reset_history(self):
        """Reset conversation history to start fresh."""
        self.chat_history_ids = None
        print("Conversation history has been reset.")

## 4. Initialize the Chatbot

In [ ]:
# Initialize the chatbot instance
chatbot = Chatbot(model, tokenizer, device)
print("Chatbot initialized successfully!")

## 5. Run Chatbot - Interactive Conversation Loop

In [ ]:
def run_chatbot():
    """
    Main function to run the interactive chatbot session.
    Handles user input and displays chatbot responses.
    """
    print("=" * 60)
    print("  Welcome! I am your AI Assistant powered by Transformers")
    print("  Type 'exit' or 'quit' to end the conversation")
    print("  Type 'reset' to start a new conversation")
    print("=" * 60)
    print()
    
    # Initial greeting from chatbot
    greeting = "Hello! I am your AI assistant. How can I help you today?"
    print(f"Chatbot: {greeting}")
    print()
    
    # Continuous conversation loop
    while True:
        try:
            # Accept user input
            user_input = input("You: ").strip()
            
            # Check for exit conditions
            if user_input.lower() in ["exit", "quit"]:
                print()
                print("Chatbot: Goodbye! It was nice talking with you. Have a great day!")
                print()
                break
            
            # Check for reset condition
            if user_input.lower() == "reset":
                chatbot.reset_history()
                print("Chatbot: Hello! How can I help you today?")
                print()
                continue
            
            # Skip empty inputs
            if not user_input:
                print("Please enter a message.")
                continue
            
            # Generate and display response
            response = chatbot.generate_response(user_input)
            print(f"Chatbot: {response}")
            print()
            
        except KeyboardInterrupt:
            print("\n\nChatbot: Conversation interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"Error: {e}")
            continue

# Run the chatbot
# Note: This will work in terminal/console environment
# For Jupyter environments, you can call run_chatbot() or use the cell below
run_chatbot()

## 6. Demo Conversation (Pre-recorded for Notebook Display)

In [ ]:
# Demo conversation for notebook demonstration purposes
# This shows sample interactions without requiring user input

demo_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Thank you"
]

print("=" * 60)
print("  DEMO CONVERSATION (Sample Interactions)")
print("=" * 60)
print()

# Reset history for clean demo
chatbot.reset_history()
print()

# Initial greeting
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print()

# Process each demo input
for user_input in demo_inputs:
    print(f"User: {user_input}")
    response = chatbot.generate_response(user_input)
    print(f"Chatbot: {response}")
    print()

## 7. How the Chatbot Works

### Pipeline Flow:

```
User Input → Tokenization → Model Processing → Response Generation → Decoding → Display Output
```

### Key Components:

1. **Model (DialoGPT)**: A transformer-based language model pre-trained on conversational data
   - Uses causal (unidirectional) attention
   - Generates text autoregressively

2. **Tokenizer**: Converts text to token IDs and vice versa
   - Handles special tokens (EOS, BOS)
   - Manages vocabulary

3. **Conversation History**: Maintains context across multiple turns
   - Previous inputs and outputs are fed back into the model
   - Creates coherent multi-turn conversations

4. **Generation Parameters**:
   - `temperature`: Controls randomness (lower = more deterministic)
   - `top_p`: Nucleus sampling for diverse outputs
   - `max_new_tokens`: Limits response length

## 8. Summary

This chatbot implementation demonstrates:

- **Model Loading**: Using `AutoModelForCausalLM` and `AutoTokenizer` from Hugging Face
- **Text Generation**: Prompt-based generation using transformer models
- **Conversation Flow**: Maintaining chat history for context-aware responses
- **User Interaction**: Console-based input/output handling
- **Clean Exit**: Proper termination on 'exit' or 'quit' command

### Model Used:
DialoGPT-medium (microsoft/DialoGPT-medium) - A 345M parameter model trained on conversational data

### Technologies:
- Python 3
- PyTorch (Deep Learning Framework)
- Hugging Face Transformers Library